# 13 — Shop-composition features with the frozen tuned LightGBM settings

**Optional, exploratory research.** Run in the matching project kernel with the saved local artifacts. This notebook refits two research models per expanding 2025 fold: the deployed longer-history **tuned** configuration with original features, and the same configuration with three additional shop-history features. Both use identical 2024-added training rows and identical evaluation rows. It never replaces the deployment, changes labels, selects cutoffs, downloads data, or scores January–March/April–June/registered outcomes.

**Rule declared before fitting:** mean fold AP must increase, and no individual fold AP may decrease by more than **0.01**. This is an arbitrary exploratory acceptance rule, not statistical significance or proof of unseen performance. Save all gains and deteriorations; do not try feature subsets/seeds until a preferred result appears. AP above 0.5 is not promised.

No earlier notebooks run. Missing or changed frozen inputs stop execution. Results live only in `research/13_shop_composition/<fingerprint>/` under the active data run; earlier results and partial runs are never overwritten or deleted.

In [ ]:
import ast
import hashlib
import json
import sys
from pathlib import Path
from importlib import metadata

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

REPOSITORY = Path.cwd().resolve()
if not (REPOSITORY / "pricecatcher_config.py").is_file():
    REPOSITORY = REPOSITORY.parent
if not (REPOSITORY / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open the FYP project or research folder.")
if str(REPOSITORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY))

FOLDS = ["fold_1", "fold_2", "fold_3"]
EXTRA = ["matched_shop_count_1w", "returning_shop_share_1w_pct", "matched_shop_change_1w_pct"]
MAX_FOLD_DECLINE = 0.01
REFERENCE_CUTOFF = 0.5  # Fixed descriptive operating point, not the later Jan–Mar deployment cutoff.


## 1. Define current/past-only features

A shop's weekly price is its median daily price. Match shops only against the **exact preceding calendar week**, within the same item, unit, food group, state, district and shop type.

| Feature | Definition |
|---|---|
| Matched-shop count | Number of currently observed shops also present the preceding week |
| Returning-shop share (%) | Matched count / current observed shops × 100 |
| Matched-shop change (%) | Change between median prices of the same matched shops in the two weeks |

No preceding-week observations means an **unavailable comparison (NaN)**. Observed weeks with no shared shops have count/share **zero**, but matched-price change remains missing. New-shop share is 100 minus returning-shop share; it is not added as a redundant fourth feature. No rows are dropped. Only each model's training-fitted imputer fills missing features. The original all-observation-median surge label remains unchanged.

The shop-week median and exact-week matching approach follows archived research 02 and the research 09 aggregation audit. Neither legacy model nor audit outcomes are used as model inputs.

In [ ]:
def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text())


def notebook_code(path):
    return "\n".join("".join(c["source"]) for c in read_json(path)["cells"] if c["cell_type"] == "code")


def shop_features(daily, group_keys, end):
    columns = group_keys + ["date", "premise_code", "price"]
    data = daily[columns].copy()
    data["date"] = pd.to_datetime(data.date)
    if (data[columns].isna().any().any() or data.date.gt(pd.Timestamp(end)).any()
            or data.duplicated(["date", "premise_code", "item_code"]).any()
            or not (np.isfinite(data.price) & data.price.gt(0)).all()):
        raise ValueError("Shop history needs unique, finite, positive observations within its cutoff.")
    data["week_start"] = data.date - pd.to_timedelta(data.date.dt.dayofweek, unit="D")
    data = data.loc[(data.week_start + pd.Timedelta(days=6)).le(pd.Timestamp(end))]
    keys = group_keys + ["week_start"]
    shops = data.groupby(keys + ["premise_code"], sort=False).price.median().rename("current_price").reset_index()
    previous = shops.rename(columns={"current_price": "previous_price"}).copy()
    previous["week_start"] += pd.Timedelta(days=7)
    pairs = shops.merge(previous, on=keys + ["premise_code"], how="left", validate="one_to_one")
    pairs["matched_current"] = pairs.current_price.where(pairs.previous_price.notna())
    result = pairs.groupby(keys, sort=False).agg(current_shops=("premise_code", "size"),
        matched_shop_count_1w=("previous_price", "count"), matched_current=("matched_current", "median"),
        matched_previous=("previous_price", "median")).reset_index()
    previous_counts = result[keys + ["current_shops"]].rename(columns={"current_shops": "previous_shops"})
    previous_counts["week_start"] += pd.Timedelta(days=7)
    result = result.merge(previous_counts, on=keys, how="left", validate="one_to_one")
    result["matched_shop_count_1w"] = result.matched_shop_count_1w.astype(float).where(result.previous_shops.notna())
    result["returning_shop_share_1w_pct"] = 100 * result.matched_shop_count_1w / result.current_shops
    result["matched_shop_change_1w_pct"] = 100 * (result.matched_current / result.matched_previous - 1)
    if np.isinf(result[EXTRA].to_numpy(dtype=float)).any():
        raise ValueError("Infinite shop-composition feature.")
    medians = data.groupby(keys, sort=False).price.median().rename("source_weekly_median").reset_index()
    return result[keys + EXTRA].merge(medians, on=keys, how="left", validate="one_to_one")


def add_shop_features(frame, features, keys):
    if frame.duplicated(keys).any() or features.duplicated(keys).any():
        raise ValueError("Duplicate local-series week keys.")
    result = frame.merge(features, on=keys, how="left", validate="one_to_one", sort=False)
    pd.testing.assert_frame_equal(result[frame.columns].reset_index(drop=True), frame.reset_index(drop=True))
    if result.source_weekly_median.isna().any():
        raise ValueError("A frozen row has no matching current-week source observations.")
    np.testing.assert_allclose(result.source_weekly_median, result.weekly_median_price, rtol=1e-9, atol=1e-9)
    return result.drop(columns="source_weekly_median")


def fit_scores(template, training, evaluation, settings, enhanced):
    pipeline = clone(template)  # Clears all fitted preprocessing and classifier state.
    extra = EXTRA if enhanced else []
    preprocess = pipeline.named_steps["preprocess"]
    preprocess.set_params(transformers=[(name, transformer, list(columns) + extra if name == "numeric" else columns)
        for name, transformer, columns in preprocess.transformers])
    def inputs(frame):
        result = frame[settings["feature_columns"] + extra].copy()
        for column in settings["categorical_features"]:
            result[column] = result[column].astype("string").fillna("__missing__").astype(str)
        return result
    pipeline.fit(inputs(training), training.surge_label.astype(int))
    scores = pipeline.predict_proba(inputs(evaluation))[:, 1]
    if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError("Invalid research scores.")
    return scores


def score_metrics(labels, scores):
    y = np.asarray(labels, dtype=int)
    alerts = scores >= REFERENCE_CUTOFF
    return {"rows": len(y), "positives": int(y.sum()), "average_precision":
        float(average_precision_score(y, scores)) if len(np.unique(y)) == 2 else np.nan,
        "precision_at_05": float(precision_score(y, alerts, zero_division=0)),
        "recall_at_05": float(recall_score(y, alerts, zero_division=0)) if y.sum() else np.nan,
        "f1_at_05": float(f1_score(y, alerts, zero_division=0)) if y.sum() else np.nan}


## 2. Verify the existing deployment and reuse 09b's training calculation

The existing pin must name longer-history **tuned** LightGBM. No pin is created and no latest-study or best-score fallback is allowed. Clone its actual fitted pipeline so model settings, seed and preprocessing match for both research fits. Verify original files, package versions and reconstructed support against the pinned study.

Only `training_table` and `prepare_training` definitions from main 09b are loaded. A wrapper derives shop features from the exact cleaned 2024–2025 daily input passed to its training-table calculation, so cleaning is not reimplemented here. Evaluation shop features use that fold's frozen daily input up to its evaluation boundary. Other weeks cannot influence an earlier feature. Prepared test/holdout outcomes are not parsed; provenance checks may hash existing upstream files.

In [ ]:
def prepare_context():
    import pricecatcher_config as cfg
    pin_path = cfg.PROJECT_ROOT / "11_deployment/deployment_selection.json"
    if not pin_path.is_file():
        raise FileNotFoundError("Restore the existing deployment pin. This experiment never creates one.")
    pin, control_dir, parent = cfg.dashboard_candidate()
    selection = parent["selection"]
    if (selection.get("family") != "LightGBM" or selection.get("training_start") != "2024-01"
            or selection.get("candidate_label") != "Longer-history LightGBM (tuned)"):
        raise ValueError("This protocol requires the existing deployed longer-history tuned LightGBM.")
    if sys.version_info[:2] != tuple(map(int, parent["python_version"].split(".")[:2])):
        raise ValueError("Use the pinned study's Python major/minor version.")
    for package in ["lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]:
        if metadata.version(package) != parent["package_versions"][package]:
            raise ValueError(f"Restore the saved {package} version.")
    protected = {str(pin_path): file_hash(pin_path), str(Path(cfg.__file__)): file_hash(cfg.__file__)}
    for path in [cfg.PROJECT_ROOT / "08_dashboard/active_deployment.json",
                 cfg.PROJECT_ROOT / "11_deployment/anomaly_selection.json"]:
        if path.is_file(): protected[str(path)] = file_hash(path)
    for path in (cfg.PROJECT_ROOT / "13_registered_future_evaluation").glob("*/registration.json"):
        protected[str(path)] = file_hash(path)
    for name, digest in parent["input_sha256"].items(): protected[str(cfg.PROJECT_ROOT / name)] = digest
    for name, digest in parent["additional_input_sha256"].items(): protected[str(cfg.resolve_input(name))] = digest
    for name, digest in parent["output_sha256"].items(): protected[str(control_dir / name)] = digest
    manifest_path = control_dir / "study_manifest.json"
    protected[str(manifest_path)] = pin["study_manifest_sha256"]
    source = REPOSITORY / "09b_longer_history_lightgbm.ipynb"
    protected[str(source)] = file_hash(source)
    functions = [node for cell in read_json(source)["cells"] if cell["cell_type"] == "code"
        for node in ast.parse("".join(cell["source"])).body
        if isinstance(node, ast.FunctionDef) and node.name in {"training_table", "prepare_training"}]
    if sorted(node.name for node in functions) != ["prepare_training", "training_table"]:
        raise ValueError("Expected the two original historical-training calculations.")
    folders = {fold: cfg.PROJECT_ROOT / "experiments" / fold / "05_modelling" for fold in FOLDS}
    settings_by_fold = {fold: read_json(folder / "experiment_settings.json") for fold, folder in folders.items()}
    for settings in settings_by_fold.values():
        if (settings["feature_columns"] != selection["feature_columns"] or settings["surge_threshold_pct"] != 10
                or settings["minimum_records"] != cfg.MIN_RECORDS or settings["minimum_premises"] != cfg.MIN_PREMISES
                or settings["minimum_history_4w"] != cfg.MIN_HISTORY_4W
                or any(settings["preparation"].get(k) != v for k, v in cfg.PREPARATION_SETTINGS.items())):
            raise ValueError("Frozen features, target, coverage or preparation differ from current settings.")
    required = [control_dir / "fold_comparison.csv", control_dir / "training_comparison.csv", control_dir / "lightgbm.joblib"]
    for folder in folders.values():
        required += [folder / "train.parquet", folder / "validation.parquet", folder / "experiment_settings.json",
                     folder.parent / "03_cleaned/daily_prices.parquet", folder.parent / "03_cleaned/cleaning_summary.json"]
    if any(str(path) not in protected for path in required):
        raise ValueError("A required fold input is not recorded by the pinned study.")
    paths = [cfg.resolve_input(name) for name in parent["additional_input_sha256"]
             if cfg.resolve_input(name).name == "selected_prices_2024.parquet"]
    if len(paths) != 1: raise ValueError("Restore the exact recorded 2024 cache; no downloads are made.")
    earlier = pd.read_parquet(paths[0], columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"])
    earlier["date"] = pd.to_datetime(earlier.date)
    if earlier.empty or not earlier.date.between("2024-01-01", "2024-12-31").all():
        raise ValueError("Expected the recorded 2024-only source.")
    namespace = {"pd": pd, "np": np, "json": json, "cfg": cfg, "folders": folders,
        "settings_by_fold": settings_by_fold, "earlier_source": earlier,
        "keys": cfg.GROUP_KEYS + ["week_start"], "features": selection["feature_columns"],
        "training_rows": [], "cleaning_rows": []}
    exec(compile(ast.Module(body=functions, type_ignores=[]), "09b training calculations only", "exec"), namespace)
    return {"cfg": cfg, "pin": pin, "control_dir": control_dir, "parent": parent,
            "protected": protected, "namespace": namespace,
            "template": joblib.load(control_dir / "lightgbm.joblib"),
            "support": pd.read_csv(control_dir / "training_comparison.csv")}


def check_protected(context):
    for name, expected in context["protected"].items():
        if file_hash(name) != expected:
            raise ValueError(f"Frozen input changed: {name}. Nothing is repaired or overwritten.")


def prepare_fold(name, context):
    ns, cfg = context["namespace"], context["cfg"]
    base_calculation = ns["training_table"]
    captured = {}
    def with_shop_history(daily, start_month):
        if start_month == "2024-01":
            captured["features"] = shop_features(daily, cfg.GROUP_KEYS, ns["training_end"])
        return base_calculation(daily, start_month)
    ns["training_table"] = with_shop_history
    try:
        training, evaluation = ns["prepare_training"](name)
    finally:
        ns["training_table"] = base_calculation
    for history, rebuilt in zip(["2025-only", "2024-added"], ns["training_rows"][-2:]):
        expected = context["support"].loc[context["support"].fold.eq(name) & context["support"].history.eq(history)]
        if len(expected) != 1 or any(str(rebuilt[k]) != str(expected.iloc[0][k])
                for k in ["rows", "positives", "first_week", "last_outcome"]):
            raise ValueError("Historical training support differs from the pinned study.")
    settings = ns["settings_by_fold"][name]
    daily = pd.read_parquet(ns["folders"][name].parent / "03_cleaned/daily_prices.parquet",
        columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"],
        filters=[("date", "<=", pd.Timestamp(settings["validation_end"]))])
    features = shop_features(daily, cfg.GROUP_KEYS, settings["validation_end"])
    keys = cfg.GROUP_KEYS + ["week_start"]
    return (add_shop_features(training, captured["features"], keys),
            add_shop_features(evaluation, features, keys), settings)


## 3. Freeze the protocol and compare once per fold

The fingerprint records the source study, fixed parameters, new features, decision rule, input hashes and calculation code. A protocol is saved **before fitting**. Completed matching results are verified and reused. A partial result folder stops execution for inspection; never delete it automatically.

Both models are fitted from scratch on each fold. Original-feature AP must reproduce the pinned control's saved fold AP within 1e-6. Report all AP changes and group counts. Precision/recall/F1 use the same descriptive **0.5** reference for both models; the later January–March deployment cutoff is not applied backwards to 2025 folds. The AP rule never selects a new deployed model.

In [ ]:
def run_experiment(context):
    cfg, parent = context["cfg"], context["parent"]
    check_protected(context)
    source = REPOSITORY / "research/13_shop_composition.ipynb"
    code_hash = hashlib.sha256(notebook_code(source).encode()).hexdigest()
    protocol = {"deployment_id": context["pin"]["deployment_id"], "source_study_sha256": context["pin"]["study_manifest_sha256"],
        "parameters": context["template"].named_steps["model"].get_params(), "extra_features": EXTRA,
        "mean_ap_gain_required": ">0", "maximum_fold_decline": MAX_FOLD_DECLINE, "cutoff_reference": REFERENCE_CUTOFF,
        "folds": FOLDS, "research_code_sha256": code_hash, "input_sha256": context["protected"],
        "status": "exploratory on previously inspected folds; no independent-test claim"}
    fingerprint = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]
    output = cfg.PROJECT_ROOT / "research/13_shop_composition" / fingerprint
    manifest_path = output / "research_manifest.json"
    if manifest_path.is_file():
        saved = read_json(manifest_path)
        if saved["protocol"] != protocol: raise ValueError("Existing protocol differs.")
        for name, expected in saved["output_sha256"].items():
            if file_hash(output / name) != expected: raise ValueError("Saved research output changed.")
        print("Reusing verified completed research; no refitting.")
        return output
    if output.exists(): raise ValueError("Partial research folder exists. Preserve and inspect it; no automatic overwrite.")
    output.mkdir(parents=True, exist_ok=False)
    (output / "protocol.json").write_text(json.dumps(protocol, indent=2))
    selected = parent["selection"]
    control = pd.read_csv(context["control_dir"] / "fold_comparison.csv")
    control = control.loc[control.method.eq(selected["comparison_method"])].set_index("fold")
    if len(control) != 3 or set(control.index) != set(FOLDS): raise ValueError("Expected exactly the pinned control's three folds.")
    results, group_rows, coverage = [], [], []
    for fold in FOLDS:
        training, evaluation, settings = prepare_fold(fold, context)
        if training.surge_label.nunique() != 2 or evaluation.surge_label.nunique() != 2:
            raise ValueError("Overall fold comparison requires both observed classes.")
        reference = control.loc[fold]
        if int(reference.rows) != len(evaluation) or int(reference.positives) != int(evaluation.surge_label.sum()):
            raise ValueError("Evaluation population differs from the pinned control.")
        predictions = evaluation[context["cfg"].GROUP_KEYS + ["week_start", "next_week_end", "surge_label"]].copy()
        row = {"fold": fold, "training_rows": len(training), "evaluation_rows": len(evaluation)}
        for enhanced, label in [(False, "control"), (True, "shop_features")]:
            scores = fit_scores(context["template"], training, evaluation, settings, enhanced)
            metrics = score_metrics(evaluation.surge_label, scores)
            if not enhanced and not np.isclose(metrics["average_precision"], reference.average_precision, rtol=1e-6, atol=1e-6):
                raise ValueError("Refitted control AP does not reproduce. Investigate before interpreting changes.")
            row[label + "_ap"] = metrics["average_precision"]
            predictions[label + "_score"] = scores
            group_rows.append({"fold": fold, "method": label, "project_group": "ALL", **metrics})
            for group, indices in evaluation.groupby("project_group", sort=True).indices.items():
                group_rows.append({"fold": fold, "method": label, "project_group": group,
                                   **score_metrics(evaluation.iloc[indices].surge_label, scores[indices])})
        row["ap_change"] = row["shop_features_ap"] - row["control_ap"]
        results.append(row)
        for split, frame in [("training", training), ("evaluation", evaluation)]:
            coverage.append({"fold": fold, "split": split, "rows": len(frame),
                "unavailable_previous_week": int(frame.matched_shop_count_1w.isna().sum()),
                "observed_zero_overlap": int(frame.matched_shop_count_1w.eq(0).sum()),
                "missing_matched_price_change": int(frame.matched_shop_change_1w_pct.isna().sum())})
        predictions.to_parquet(output / (fold + "_predictions.parquet"), index=False)
        print(fold, "AP change:", round(row["ap_change"], 6))
    comparison = pd.DataFrame(results)
    summary = {"control_mean_ap": float(comparison.control_ap.mean()),
        "shop_features_mean_ap": float(comparison.shop_features_ap.mean()),
        "mean_ap_change": float(comparison.ap_change.mean()), "worst_fold_change": float(comparison.ap_change.min()),
        "meets_exploratory_rule": bool(comparison.ap_change.mean() > 0 and comparison.ap_change.min() >= -MAX_FOLD_DECLINE)}
    comparison.to_csv(output / "fold_comparison.csv", index=False)
    pd.DataFrame([summary]).to_csv(output / "fold_summary.csv", index=False)
    pd.DataFrame(group_rows).to_csv(output / "group_metrics.csv", index=False)
    pd.DataFrame(coverage).to_csv(output / "feature_coverage.csv", index=False)
    pd.DataFrame(context["namespace"]["training_rows"]).to_csv(output / "training_support.csv", index=False)
    check_protected(context)
    if hashlib.sha256(notebook_code(source).encode()).hexdigest() != code_hash:
        raise ValueError("Research calculation changed during execution.")
    record = {"status": "completed", "protocol": protocol, "summary": summary,
        "completed_at": pd.Timestamp.now(tz="Asia/Kuala_Lumpur").isoformat(),
        "python_version": sys.version.split()[0], "package_versions": parent["package_versions"],
        "deployment_changed": False, "target_changed": False, "later_period_scored": False,
        "output_sha256": {p.name: file_hash(p) for p in output.iterdir() if p.is_file()}}
    with manifest_path.open("x") as handle: json.dump(record, handle, indent=2)
    return output


## 4. Run and interpret

Run All on your Mac in the saved environment. The cloud checkout has no real artifacts; validation here is static and synthetic only. This performs six research fits and may take time. It reuses completed identical results, rather than trying again for a better score.

Share `fold_comparison.csv`, `fold_summary.csv` and `feature_coverage.csv`. A passed rule supports only a promising exploratory feature change; a failed rule is useful evidence too. Examine all group results and missingness. Do not promote this candidate, change the dashboard cutoff, or claim a higher unseen/test AP from this experiment. No new deployment model is saved; fold predictions and full protocol remain available for reproduction.

In [ ]:
context = prepare_context()
OUTPUT = run_experiment(context)
print("Results:", OUTPUT)
print("Same target and rows; deployment and later-period evaluations unchanged.")
for name in ["fold_comparison.csv", "fold_summary.csv", "feature_coverage.csv", "group_metrics.csv"]:
    print(name)
    display(pd.read_csv(OUTPUT / name))
